### Prepare DATA 

A. Oetjens, March 2025  
*************************************
- Create all the necessary data file structure from BGC-Argo float data
- Sanity checks but no plotting

Input:
- SONIF floats
- pythargo_utils.py

In [1]:
# Load modules
import sys 
import os

# where are we?
#print(os.getcwd())

# path to where the datafiles live
path_main = '/../../../g/data/jk72/ao2582/FOSOR/scripts/'
sys.path.append(os.path.abspath(path_main))

# Import from other scipts
from pythargo_utils import *
print(path_g)

version = 'qcv5' 
print(version)

ds_master = xr.open_dataset(path_g + 'FOSOR/' + 'master_v2.nc')
ds_floats = ds_master
ds_floats['sonif'] = ('n_prof', ds_floats.sonif.data.astype(str))
ds_floats['zone'] = ('n_prof', ds_floats.zone.data.astype(str))
ds_floats['month'] = ('n_prof', ds_floats.month.data.astype(int))

CbPM_argo -- imported
chl_box  --- imported
/../../../g/data/jk72/ao2582/
qcv5


#### Part 1: Create BGC-Argo dataset for Southern Ocean floats (SOFs)
*************************

- Data selection, based on lat/lon
- Data curation (QC flags), smoothing of bbp and chl
- Interpolation of data on 1 dbar grid
- group in zones, exclude suspicious profile, exclude no zones (nz)
- Koestner computation


In [ ]:
### 1. create RAW bgc-argo dataset based on region etc.  ###

'''
this combines the different profiles of all floats in the specified area
'''

# Interpolation of data
fosor ='SOATLANTIC'

chl_region = chl_box(fosor, chl_contour=False)[0]

argo_df =  create_argo_df(chl_region,path)
argo_dict = interp_argo(argo_df)

argo_DS = bgcalc_argo(argo_dict,fosor)

ds_floats = xr.concat([*argo_DS.values()],dim='n_prof')
ds_floats.to_netcdf('SONIF_'+str(fosor)+ 'raw.nc', encoding={'time':{'dtype':'object'},'sonif':{'dtype':'str'},'zone':{'dtype':'str'}})


# can also perform in one go, bu if it is a lot of profiles this requires a lot of emorey, so a bit safer to perform one after an other
#ds_floats, ds_dict = build_argo(fosor, argo_df, to_netcdf=False)


In [ ]:
# open nc file with floats 
fosor ='SOATLANTIC'
ds_floats = xr.open_dataset(path_sonif+ 'SONIF_SOATLANTICraw.nc')

ds_floats['sonif'] = ('n_prof', ds_floats.sonif.data.astype(str))
ds_floats['zone'] = ('n_prof', ds_floats.zone.data.astype(str))
ds_floats['month'] = ('n_prof', ds_floats.month.data.astype(int))

# exclude profiles with bad data
exclude_profs = []

# exclude data from specific regions
filtered_ds = ds_floats.where((~ds_floats['zone'].isin(['sz','nz']))& (~ds_floats['n_prof'].isin(exclude_profs)), drop=True) 
filtered_ds['month'] = filtered_ds['month'].astype(int)
ds_floats = filtered_ds

# create list of zones
zones = np.unique(ds_floats.zone.data)

#### Compute POC using Koestner algorithm 

In [ ]:
# Data prep for Koestner algorithm
# - background correction, chl correction: smoothing

poc_koest = np.ones((1501,0))

ds_nonan = ds_floats.where(ds_floats.sel(pressure=slice(850, 900)).chla.notnull().any(dim='pressure'), drop=True)
for nprof in ds_nonan.n_prof:
    ds = ds_floats.sel(n_prof = nprof)
    chl_dc = ds.chla - ds.chla.sel(pressure = np.arange(850,900), method='nearest').quantile(1/5, dim = 'pressure')
    print(nprof.data)
    chl_corr = chl_dc.where((chl_dc > ds.chla.sel(pressure = np.arange(850,900), method='nearest').quantile(1/5, dim ='pressure')) &
                        (chl_dc.pressure<chl_dc.pressure.where(abs(chl_dc.sel(pressure=np.arange(100,900)).differentiate(coord='pressure').differentiate(coord='pressure')).rolling(pressure=20,center=True, min_periods=10).max() <= 
                                                               abs(chl_dc.differentiate(coord='pressure').differentiate(coord='pressure').sel(pressure = np.arange(850,900), method='nearest').quantile(0.95))).min()), 0)/3.6*2
    
    koest23 = Koest23_modelB_700p(ds.bbp700_bs.data, chl_corr.data, alignment=0.9, bias='power', alpha=0.125)
    poc_koest = np.c_[poc_koest, koest23[0]]

ds_nonan['poc_koest_bc'] = (('pressure', 'n_prof'), poc_koest)

ds_floats = ds_nonan

print('poc_koest_bc  --  DONE')

#### Compute NPP using ARTEAGA

Arteaga et al. (2022; https://zenodo.org/record/6599224#.Y2O_duzMKyt)

In [ ]:
# in case there was no npp calculations yet

ds_new = ds_floats.copy()
par = np.array([])
fill_npp = np.ones((1501,0))

for nprof in ds_new.n_prof:
    ds = ds_new.sel(n_prof=nprof)
    lat = ds.lat
    lon = ds.lon
    ## 25.10.2024 aoetjens: set values within uncertainty to zero (below 200m)
    chla = ds.chla
    # Prepare input for ARTEAGA CbPM for upper 200m
    # Get surfaace PAR from satellite according to month, lat, lon of observation
    month = str(ds.time.dt.month.data)
    
    par_oc = glob.glob(path_g + 'AQUA_MODIS/PAR/*'+ month+'_par.nc')[0]
    dpar = xr.open_dataset(par_oc) # [einstein m^-2 day^-1]
    irr = dpar.sel(lat=ds.lat.data, method='nearest').sel(lon=ds.lon.data, method='nearest').par.data
    dpar.close()
    par = np.append(par, irr)
    # Chlorophyll data of upper 200m (skip first 10 m)
    chl_z = ds.chla.data[10:210]
    Cphyto_z = ds.cphyto.data[10:210] # this is from Johnson conversion
    # call CbPM function to compute NPP 
    [pp_z,mu_z,par_z,prcnt_z,nutTempFunc_z,IgFunc_z] = cbpm_bgcfloats(chl_z, Cphyto_z, irr, ds.time.dt, ds.lat)
    # store NPP andfill array with last value to have same shape
    filler = np.ones(len(chla))
    filler[0:200] = pp_z
    filler[200:] = pp_z[-1]
 
    fill_npp = np.c_[fill_npp, filler] # [mg C m^-2 d^-1]


In [ ]:
ds_floats.to_netcdf('SONIF_SOATLANTIC_bon_qcv5.nc', encoding={'time':{'dtype':'object'},'sonif':{'dtype':'str'},'zone':{'dtype':'str'}})
